In [3]:
!pip install -q -U langchain langchain-google-genai langchain-community faiss-cpu pypdf

In [31]:
import os
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
os.environ["GOOGLE_API_KEY"]=userdata.get("GEMINI_API_KEYS")
EMB_MODEL="gemini-embedding-001"
MODEL="gemini-3.5-flash-lite"

In [32]:
#1.Prepare Document
#1.1 Upload Document
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print(f"PDF:{pdf_name} uploaded sucessfully")

Saving Customer_Support_RAG_Knowledge_Base.pdf to Customer_Support_RAG_Knowledge_Base (1).pdf
PDF:Customer_Support_RAG_Knowledge_Base (1).pdf uploaded sucessfully


In [33]:
#1.2 Extract Text from Document
from langchain_community.document_loaders import PyPDFLoader
docs=PyPDFLoader(pdf_name).load()
print("NUMBER OF PAGES: ",len(docs))
text=""
for d in docs:
  text+=d.page_content+"\n"
print("NUMBER OF CHARACTERS: ",len(text))
print("CHECK POINT: EXTRACTION OF TEXT DONE!!")

NUMBER OF PAGES:  1
NUMBER OF CHARACTERS:  3329
CHECK POINT: EXTRACTION OF TEXT DONE!!


In [34]:
#1.3 Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=50)
chunks=splitter.split_documents(docs)
print(f"CHECKPOINT:CHUNKING DONE HAVING {len(chunks)} NUMBER OF CHUNKS")

CHECKPOINT:CHUNKING DONE HAVING 5 NUMBER OF CHUNKS


In [35]:
#1.4 Embeddings vectors and faiss
from langchain_community.vectorstores import FAISS
embeddings=GoogleGenerativeAIEmbeddings(model=EMB_MODEL)
vectorstore=FAISS.from_documents(chunks,embeddings)

In [37]:
#1.5retrieving best chunk
retriever=vectorstore.as_retriever(search_kwargs={'k':3})
found=retriever.invoke("Tell about technical issues")
print("CHECK POINT:FOUND RELAVENT CHUNKS")

"""for d in found:
  print(d.page_content)
  print('-'*60)

#TO KNOW THE SCORES:(method 2)
for d,score in vectorstore.similarity_search_with_score("Tell about technical issues",3):
  print(f"SCORE:{score:.3f}---------CHUNK:{d.page_content}")
  print('-'*60)"""

CHECK POINT:FOUND RELAVENT CHUNKS


'for d in found:\n  print(d.page_content)\n  print(\'-\'*60)\n\n#TO KNOW THE SCORES:(method 2)\nfor d,score in vectorstore.similarity_search_with_score("Tell about technical issues",3):\n  print(f"SCORE:{score:.3f}---------CHUNK:{d.page_content}")\n  print(\'-\'*60)'

In [41]:
llm=ChatGoogleGenerativeAI(model=MODEL,
                           temperature=0.2,
                           max_output_tokens=1500)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
prompt=ChatPromptTemplate.from_messages([
    ("system","You are a PDF assistant. "
     "Answer the user's request using the provided document context. "
     "For questions, give an accurate answer based only on the context. "
     "For summarization requests, summarize the information present in the context clearly and concisely. "
     "Do not invent or add information that is not present in the context. "
     "If the provided context does not contain enough information to answer the request, say: "
     "'I could not find enough information in the document.'"
     "REPLY STARTING WITH HI I AM YOUR CUSTOMER SUPPORT AGENT"),
    ("human","CONTEXT FROM THE DOCUMENT:\n\n{context}\n\nQUESTION: {question}")
])
def format_docs(docs):
  return "\n\n------------------\n\n".join(d.page_content for d in docs)
rag_chain=(
    {'context':retriever|format_docs,'question':RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("NOTE:ENTER bye,exit,quit to terminate session")
while True:
  user_input=input("AI:Enter query: ")
  if user_input.lower() in ['quit','bye','exit']:
    print("EXITED...THANK YOU")
    break
  response=rag_chain.invoke(user_input)
  print(f"AI:{response}")

NOTE:ENTER bye,exit,quit to terminate session
AI:Enter query: summarize pdf


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI:HI I AM YOUR CUSTOMER SUPPORT AGENT. Based on the provided document, here is a summary of the information:

* **Product Information:** The SmartBook Pro 14 is a 14-inch productivity laptop featuring 16 GB RAM, 512 GB SSD, and integrated graphics, suited for programming, office work, and general productivity.
* **Technical Troubleshooting:** 
  * For a **black display**, check the charging indicator, connect an external display, perform a display reset, or contact support if the issue persists.
  * For **overheating**, place the laptop on a hard flat surface, keep ventilation clear, close unnecessary apps, and restart.
* **Refund Policy:** 
  * Refund requests must be submitted within 7 days of delivery for eligible products in acceptable condition with accessories. 
  * Refunds are initiated to the original payment method after inspection and approval.
  * Non-refundable cases include products outside the refund period or damaged by customer misuse.
* **Account Help:** 
  * For a **

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI:HI I AM YOUR CUSTOMER SUPPORT AGENT

Based on the provided document, here are the details of the refund policy:

* **Refund Eligibility:** Refund requests can be submitted within 7 days of delivery for an eligible product returned in acceptable condition with included accessories.
* **Non-refundable Cases:** Products outside the stated refund period or damaged through customer misuse may not qualify for a refund.
* **Refund Processing:** After the returned product is inspected and approved, the refund is initiated to the original payment method. The processing time depends on the payment provider.
AI:Enter query: explain whole pdf in 10lines


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AI:HI I AM YOUR CUSTOMER SUPPORT AGENT. Here is a summary of the document:

1. It provides details on the SmartBook Pro 14, a 14-inch productivity laptop with 16 GB RAM and 512 GB SSD.
2. It covers troubleshooting for technical issues like a black laptop display and laptop overheating.
3. It outlines refund policies, stating refunds are initiated to the original payment method after inspection and approval.
4. Non-refundable cases include products outside the refund period or damaged by customer misuse.
5. Refund requests must be submitted within 7 days of delivery for eligible products in acceptable condition.
6. Account help includes instructions for password resets and handling temporarily locked accounts after repeated failed logins.
7. It defines when to escalate issues if documented troubleshooting fails.
8. It details a 12-month limited standard warranty covering manufacturing defects under normal use.
9. Warranty claims require a purchase invoice, serial number, and an issue de